In [70]:
# ==============================================================================
# PIPELINE DE EXTRAÇÃO, HIGIENIZAÇÃO E SALVAMENTO DA BASE LIMPA (NHIS 2025)
# ==============================================================================
import numpy as np
import pandas as pd

YES_NO = {
    1 : "SIM",
    2 : "NÃO",
    7 : np.nan,
    8 : np.nan,
    9 : np.nan
}

QTD = {
    1: "1",
    2: "2",
    3: "3+",
    4: "INCERTO"
}

# 1. Dicionário Consolidado das Variáveis Selecionadas (101 Features + Controles)
MAPA_VARIAVEIS_PAPER = {
    # Controles amostrais
    "HHX": "id_domicilio", 
    "WTFA_A": "peso_amostral",
    #dados pessoais
    "SEX_A": "sexo",
    "URBRRL23": "classificacao_urbano_rural",
    "REGION": "regiao_geografica",
    "AGE65": "faixa_65_mais",
    #dados família
    "MARITAL_A": "estado_civil_declarado",
    "MARSTAT_A": "estado_civil",
    "PARSTAT_A": "status_parental", 
    "SPOUSESEX_A": "sexo_conjuge",
    "SPOUSEP_A": "separacao_legal_conjugal",
    "SPOUSWRK_A": "cojuge_trabalha",
    "OVER65FLG_A": "flg_idoso_familia",
    "PCNTADLT_A": "qtd_adultos_familia",
    "PCNTKIDS_A": "qtd_crianças_familia",
    "MAXEDUCP_A": "escolaridade_maxima_familia",
    
    #moradia
    "HOUYRSLIV_A": "tempo_moradia_anos",
    #"HOUSECOST_A": "dificuldade_custo_moradia", #não estava no paper e não tem em 2025
    "HOUTENURE_A": "tipo_posse_imovel", 
    "YRSINUS_A": "qtd_anos_estados_unidos",


#######################################################################################


    #economicos
    "FLUNCH12M1_A":"merenda_escolar_gratuita",
    "RATCAT_A": "categoria_razao_pobreza",
    "POVRATTC_A": "razao_renda_pobreza",
    "INCINTER_A":"renda_investimentos",
    "HISTOPJOB_A":"perdeu_plano_demitido_ou_trocou_emprego",
    "PAYWORRY_A": "estresse_financeiro_medico",
    "PAYNOBLLNW_A": "dividas_medicas_em_aberto",
    "RSNHICOST_A": "sem_plano_por_custo_inacessivel",

    #plano de saude
    "OPDEDUC_A": "plano_estadual_dedutivel",
    "PRDEDUC1_A": "plano_privado_dedutivel",
    "PRDEDUC2_A": "plano_privado_dedutivel2",
    "VAHOSP_A": "plano_militar_saude",
    "PLN1PAY2_A": "plano_patrao_sindicato_pagam",
    "HINOTMYR_A": "plano_meses_sem", # HINOTYR_A
    

#######################################################################


    #saude geral
    "PHSTAT_A": "autoavaliacao_saude_geral", 
    "BMICAT_A": "categoria_imc",
    "WEIGHTLBTC_A": "peso_libras",
    "LSATIS4_A": "satisfacao_com_a_vida",
    "SOCWRKLIM_A": "limitacao_trabalho_por_saude",
    
    "ASTILL_A": "asma_ativa_atualmente",
    "HYPEV_A": "historico_hipertensao",
    "CHLEV_A": "historico_colesterol",
    "CHLMED_A": "remedio_colesterol",
    "DIBINSSTYR_A": "interrompeu_insulina_prm_ano",
    "ARTHEV_A": "historico_artrite",
    "DIBEV_A": "historico_diabetes",
    "DIBPILL_A": "med_diabetes_oral", # no paper usou dibpill mas tem vários outros tratamentos

    "PAIAPG3M_A": "dor_abdominal_cronica",

    
    "TBIHLSBMC_A": "concussao_sintomas_pos_trauma", #não tem no & artigo não tem em 2025
    #"INJFALL_A": "lesao_decorrente_queda",#não tem no artigo & não tem em 2025
    #"INJFALLHOM_A": "queda_ocorrida_em_casa", #não tem no artigo & não tem em 2025

    #"REPWRKCAUS_A": "ler_dort_causada_no_trabalho", não tem em 2025
    #"REPSTRAIN_A": "lesao_por_esforco_repetitivo", não tem em 2025
    #"REPLIMIT_A": "limitacao_por_ler_dort", não tem em 2025

    #câncer
    "CERVIAGETC_A": "idade_cancer_colo_utero",
    "PANCRAGETC_A": "idade_cancer_pancreas", #
    "ESOPHAGETC_A": "idade_cancer_esofago", #
    "LUNGAGETC_A": "idade_cancer_pulmao",
    "LIVERAGETC_A": "idade_cancer_figado",
    "SKNDKAGETC_A": "idade_cancer_pele_indeterminado",
    "THYROAGETC_A": "idade_cancer_tireoide",
    "STOMAAGETC_A": "idade_cancer_estomago",


    #vacinação

    "SHTFLUY_A": "ano_vacina_gripe",
    "SHTFLU12M_A": "vacina_gripe_ultimos_12m",

    "SHTPNEUNB_A": "qtd_vacina_pneumonia",
    "SHTTETANUS_A": "vacina_tetanus_10a",#


    #"INJWRKDYTC_A": "problema_saude_dias_sem_trabalhar"



###################################################################################################
    #Uso de recursos de saude ####################################################################

    "URGCC12MTC_A": "visitas_pronto_atendimento_12m",
    "EMERG12MTC_A": "visitas_emergencia_hospitalar_12m",
    "EYEEX12M_A": "visitas_exame_vista_12m",
    "WELLVIS_A": "tempo_ultimo_checkup_geral",
    "WELLNESS_A": "ultima_visita_foi_checkup", #meio redundante

    #"INJSAWDOC_A": "atendimento_medico_por_lesao", não tem e, 2025


##################################################################################################




    #Ansiedade, Depressão e seus Sintomas ########################################################

    "DEPEV_A": "historico_depressao",
    "DEPMED_A": "remedio_depressão", #pode enviesar muito!
    "DEPFREQ_A": "freq_depressao",
    "DEPLEVEL_A": "intensidade_depressao",
    "PHQ81_A": "freq_perda_interesse_2sem",
    "PHQ82_A": "freq_sentiu_deprimido_2sem",
    #"HOPELESS_A": "k6_freq_desesperanca", #infelizmente não tem 2025. PHQ82_A equivalente?
    #"WORTHLESS_A": "k6_freq_desvalorizacao", #infelizmente não tem 2025. PHQ82_A equivalente?
    "GAD72_A": "freq_preocupacao_incontrolavel_2sem",
    "PHQ83_A": "freq_dificuldade_dormir_2sem",
    #"SLPFLL_A": "dificuldade_adormecer", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPSTY_A": "dificuldade_manter_sono", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPREST_A": "acorda_descansado", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    "PHQ84_A": "freq_cansaso_2sem",
    #"FGEFRQTRD_A": "frequencia_fadiga_3m", #infelizmente não tem no paper e 2025. PHQ84_A equivalente?
    "PHQ85_A": "freq_hiporexia_compulsao_alimentar_2sem",
    "PHQ86_A": "freq_autoavaliacao_negtiva_2sem", #auto estima, etc...
    "PHQ87_A": "freq_dificuldade_concentracao_2sem",
    #"SAD_A": "k6_freq_tristeza_profunda", #infelizmente não tem 2025. PHQ82_A equivalente?
    "LONELY_A": "freq_solitário", #não está no paper mas deveria
    #"EFFORT_A": "k6_freq_esforco_extremo", não está no paper. Não achei equivalente
    #"PHQCAT_A": "escala_sintomas_depressão", #pode enviesar muito!
    
    "COGFRQDFF_A": "frequencia_falha_memoria", 
    "COGTYPEDFF_A": "tipo_dificuldade_cognitiva",
    "COGMEMDFF_A": "qtd_dificuldade_cognitiva",

    "ANXEV_A": "historico_transtorno_ansiedade",
    #"ANXMED_A": "med_ansiedade", #pode enviesar muito!
    "ANXFREQ_A": "freq_ansiedade",
    "ANXLEVEL_A": "intensidade_ansiedade",

    "GAD71_A": "freq_ansiedade_2sem",
    #"NERVOUS_A": "k6_freq_nervosismo",
    "GAD72_A": "freq_preocupacao_descontrolada_2sem",
    "GAD73_A": "freq_preocupacao_desnecessaria_2sem",
    "GAD74_A": "freq_dificuldade_descansar_2sem",
    "GAD75_A": "freq_agitacao_inquieta", # não tinha no paper mas é de ansiedade

    #"GADCAT_A": "escala_sintomas_ansiedade", #pode enviesar muito!
    #"K6SPD_A": "sofrimento_psicologico_k6", #não tem no paper nem 2025. Pode enviesar?
    ##############################################################################################################
    
    # Segurança alimentar: INFELIZMENTE NÃO TEM EM 2025
    #"FDSLESS_A": "comeu_menos_que_devia", 
    #"FDSWEIGHT_A": "perdeu_peso_restricao_alimentar", 
    #"FDSSKIP_A": "diminuiu_ou_pulou_refeicoes",
    #"FDSHUNGRY_A": "passou_fome_por_falta_dinheiro",
    
    #################################################################################################################
    
    #Dificuldade de andar
    "NOEQWLK13M_A": "andar_dificuldade_500m_sem_aparelho",
    "NOEQSTEPS_A": "andar_dificuldade_degraus_sem_aparelho",
    "EQWLK13M_A": "andar_dificuldade_500m_com_aparelho",
    "EQSTEPS_A": "andar_dificuldade_degraus_com_aparelho",
    "PERASST_A": "andar_necessita_ajuda_outra_pessoa",
    
    #####################################################################
    
    #
    
    "SUPPORT_A": "suporte_social", #não tem no artigo mas deveria

    #SCONNECT1_A    Number of times you get together with people you are close to 
    #SCONNECT2_A    Number of times you talk to people on telephone or by video 
    #SCONNECT3_A    Number of times you attend religious services 
    #SCONNECT4_A    Number of times you attend meetings of clubs or organizations 
    #PARENTCON1_A   How often receive social/emotional support parenting 
    #PARENTCON2_A   How difficult are day-to-day demands parenting 
    #PARENTCON3_A   How often do you talk, chat, or connect with other parents 
    
    #################################################################################################
    #Uso de cigarro

    "SMKNOW_A": "fuma_cigarro_atualmente",
    "CIGNOW_A": "qtd_cigarros_dia",
    "SMK30D_A": "dias_fumados_mes",
    "CIG30D_A": "qtd_cigarro_mes",
    "ECIGNOW_A": "usa_cigarro_eletronico_atualmente",
    "SMKEV_A": "ja_fumante", # na verdade foi perguntado já fumou 100 cigarros na vida. Isso categoriza como fumante? 

    
    
}

colunas_para_ler = list(MAPA_VARIAVEIS_PAPER.keys())
print(f"Colunas a serem carregadas: {len(colunas_para_ler)}")

# 2. Leitura otimizada direto do CSV
df_bruto = pd.read_csv("dados/adult25.csv", usecols=colunas_para_ler, low_memory=False)
print(f"Base carregada: {df_bruto.shape[0]:,} linhas e {df_bruto.shape[1]} colunas.")

# 3. Construção do Desfecho Clínico (target_medicacao)
# 1 = Sim, 2 = Não, outros = Indeterminado
#cond_sim = (df_bruto["DEPMED_A"] == 1) | (df_bruto["ANXMED_A"] == 1)
#cond_nao = (df_bruto["DEPMED_A"] == 2) & (df_bruto["ANXMED_A"] == 2)
#df_bruto["target_medicacao"] = np.where(cond_sim, 1, np.where(cond_nao, 0, np.nan))

# 4. Aplicação dos Nomes Padronizados em Português
df_limpo = df_bruto.rename(columns=MAPA_VARIAVEIS_PAPER)

# 5. Exportação para CSV limpo
nome_arquivo_saida = "dado_ingerido/adult25_limpo_selecionadas.csv"
df_limpo.to_csv(nome_arquivo_saida, index=False)

print(f"\nBase limpa exportada como '{nome_arquivo_saida}' com sucesso!")
print(f"Dimensões finais: {df_limpo.shape[0]:,} participantes e {df_limpo.shape[1]} atributos.")
print("\nDistribuição da variável dependente (target_medicacao):")
#print(df_limpo["target_medicacao"].value_counts(dropna=False, normalize=True).round(4) * 100)

Colunas a serem carregadas: 100
Base carregada: 24,215 linhas e 100 colunas.

Base limpa exportada como 'dado_ingerido/adult25_limpo_selecionadas.csv' com sucesso!
Dimensões finais: 24,215 participantes e 100 atributos.

Distribuição da variável dependente (target_medicacao):


In [71]:
df_bruto.head()

,RATCAT_A,YRSINUS_A,MARSTAT_A,SPOUSWRK_A,SPOUSESEX_A,BMICAT_A,WEIGHTLBTC_A,URGCC12MTC_A,EMERG12MTC_A,MAXEDUCP_A,...,DIBEV_A,ASTILL_A,CHLMED_A,CHLEV_A,HYPEV_A,LSATIS4_A,PHSTAT_A,WTFA_A,HHX,POVRATTC_A
0,6,NaN,1,2.0,1.0,3,171,0,0,4,...,2,NaN,NaN,2,2,2,3,10636.862,H012128,1.58
1,12,NaN,5,NaN,NaN,2,140,0,0,3,...,1,NaN,2.0,1,1,1,4,2996.860,H000617,4.32
2,1,NaN,7,NaN,NaN,3,175,0,0,2,...,2,NaN,NaN,2,2,2,3,29032.445,H033243,0.18
3,14,NaN,1,2.0,2.0,3,215,0,0,5,...,1,NaN,1.0,1,1,1,4,7875.107,H049140,6.32
4,13,NaN,7,NaN,NaN,4,198,0,0,8,...,2,NaN,1.0,1,1,2,2,4824.620,H012797,4.90


In [72]:
def porcentagem_nan(df):
    df_analisado = df.isna()
    porcentagens = {}
    for coluna in df_analisado.columns:
        porcentagens[coluna] = float(df_analisado[coluna].sum() / len(df_analisado) * 100)
    porcentagens = sorted(porcentagens.items(),key=lambda item: item[1], reverse=True)
    return print(porcentagens)

#porcentagem_nan(df_bruto)

In [73]:
# ==============================================================================
# PIPELINE DE EXTRAÇÃO, HIGIENIZAÇÃO E SALVAMENTO DA BASE LIMPA (NHIS 2025)
# ==============================================================================
import numpy as np
import pandas as pd


# 1. Dicionário Consolidado das Variáveis Selecionadas (101 Features + Controles)
MAPA_VARIAVEIS_PAPER = {
    # Controles amostrais
    "HHX": "id_domicilio", 
    "WTFA_A": "peso_amostral",
    #dados pessoais
    "SEX_A": "sexo",
    "URBRRL23": "classificacao_urbano_rural",
    "REGION": "regiao_geografica",
    "AGE65": "faixa_65_mais",
    #dados família
    "MARITAL_A": "estado_civil_declarado",
    "MARSTAT_A": "estado_civil",
    "PARSTAT_A": "status_parental", 
    "SPOUSESEX_A": "sexo_conjuge",
    "SPOUSEP_A": "separacao_legal_conjugal",
    "SPOUSWRK_A": "cojuge_trabalha",
    "OVER65FLG_A": "flg_idoso_familia",
    "PCNTADLT_A": "qtd_adultos_familia",
    "PCNTKIDS_A": "qtd_crianças_familia",
    "MAXEDUCP_A": "escolaridade_maxima_familia",
    
    #moradia
    "HOUYRSLIV_A": "tempo_moradia_anos",
    #"HOUSECOST_A": "dificuldade_custo_moradia", #não estava no paper e não tem em 2025
    "HOUTENURE_A": "tipo_posse_imovel", 
    "YRSINUS_A": "qtd_anos_estados_unidos",


#######################################################################################


    #economicos
    "FLUNCH12M1_A":"merenda_escolar_gratuita",
    "RATCAT_A": "categoria_razao_pobreza",
    "POVRATTC_A": "razao_renda_pobreza",
    "INCINTER_A":"renda_investimentos",
    "HISTOPJOB_A":"perdeu_plano_demitido_ou_trocou_emprego",
    "PAYWORRY_A": "estresse_financeiro_medico",
    "PAYNOBLLNW_A": "dividas_medicas_em_aberto",
    "RSNHICOST_A": "sem_plano_por_custo_inacessivel",

    #plano de saude
    "OPDEDUC_A": "plano_estadual_dedutivel",
    "PRDEDUC1_A": "plano_privado_dedutivel",
    "PRDEDUC2_A": "plano_privado_dedutivel2",
    "VAHOSP_A": "plano_militar_saude",
    "PLN1PAY2_A": "plano_patrao_sindicato_pagam",
    "HINOTMYR_A": "plano_meses_sem", # HINOTYR_A
    

#######################################################################


    #saude geral
    "PHSTAT_A": "autoavaliacao_saude_geral", 
    "BMICAT_A": "categoria_imc",
    "WEIGHTLBTC_A": "peso_libras",
    "LSATIS4_A": "satisfacao_com_a_vida",
    "SOCWRKLIM_A": "limitacao_trabalho_por_saude",
    
    "ASTILL_A": "asma_ativa_atualmente",
    "HYPEV_A": "historico_hipertensao",
    "CHLEV_A": "historico_colesterol",
    "CHLMED_A": "remedio_colesterol",
    "DIBINSSTYR_A": "interrompeu_insulina_prm_ano",
    "ARTHEV_A": "historico_artrite",
    "DIBEV_A": "historico_diabetes",
    "DIBPILL_A": "med_diabetes_oral", # no paper usou dibpill mas tem vários outros tratamentos

    "PAIAPG3M_A": "dor_abdominal_cronica",

    
    "TBIHLSBMC_A": "concussao_sintomas_pos_trauma", #não tem no & artigo não tem em 2025
    #"INJFALL_A": "lesao_decorrente_queda",#não tem no artigo & não tem em 2025
    #"INJFALLHOM_A": "queda_ocorrida_em_casa", #não tem no artigo & não tem em 2025

    #"REPWRKCAUS_A": "ler_dort_causada_no_trabalho", não tem em 2025
    #"REPSTRAIN_A": "lesao_por_esforco_repetitivo", não tem em 2025
    #"REPLIMIT_A": "limitacao_por_ler_dort", não tem em 2025

    #câncer
    "CERVIAGETC_A": "idade_cancer_colo_utero",
    "PANCRAGETC_A": "idade_cancer_pancreas", #
    "ESOPHAGETC_A": "idade_cancer_esofago", #
    "LUNGAGETC_A": "idade_cancer_pulmao",
    "LIVERAGETC_A": "idade_cancer_figado",
    "SKNDKAGETC_A": "idade_cancer_pele_indeterminado",
    "THYROAGETC_A": "idade_cancer_tireoide",
    "STOMAAGETC_A": "idade_cancer_estomago",


    #vacinação

    "SHTFLUY_A": "ano_vacina_gripe",
    "SHTFLU12M_A": "vacina_gripe_ultimos_12m",

    "SHTPNEUNB_A": "qtd_vacina_pneumonia",
    "SHTTETANUS_A": "vacina_tetanus_10a",#


    #"INJWRKDYTC_A": "problema_saude_dias_sem_trabalhar"



###################################################################################################
    #Uso de recursos de saude ####################################################################

    "URGCC12MTC_A": "visitas_pronto_atendimento_12m",
    "EMERG12MTC_A": "visitas_emergencia_hospitalar_12m",
    "EYEEX12M_A": "visitas_exame_vista_12m",
    "WELLVIS_A": "tempo_ultimo_checkup_geral",
    "WELLNESS_A": "ultima_visita_foi_checkup", #meio redundante

    #"INJSAWDOC_A": "atendimento_medico_por_lesao", não tem e, 2025


##################################################################################################




    #Ansiedade, Depressão e seus Sintomas ########################################################

    "DEPEV_A": "historico_depressao",
    "DEPMED_A": "remedio_depressão", #pode enviesar muito!
    "DEPFREQ_A": "freq_depressao",
    "DEPLEVEL_A": "intensidade_depressao",
    "PHQ81_A": "freq_perda_interesse_2sem",
    "PHQ82_A": "freq_sentiu_deprimido_2sem",
    #"HOPELESS_A": "k6_freq_desesperanca", #infelizmente não tem 2025. PHQ82_A equivalente?
    #"WORTHLESS_A": "k6_freq_desvalorizacao", #infelizmente não tem 2025. PHQ82_A equivalente?
    "GAD72_A": "freq_preocupacao_incontrolavel_2sem",
    "PHQ83_A": "freq_dificuldade_dormir_2sem",
    #"SLPFLL_A": "dificuldade_adormecer", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPSTY_A": "dificuldade_manter_sono", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPREST_A": "acorda_descansado", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    "PHQ84_A": "freq_cansaso_2sem",
    #"FGEFRQTRD_A": "frequencia_fadiga_3m", #infelizmente não tem no paper e 2025. PHQ84_A equivalente?
    "PHQ85_A": "freq_hiporexia_compulsao_alimentar_2sem",
    "PHQ86_A": "freq_autoavaliacao_negtiva_2sem", #auto estima, etc...
    "PHQ87_A": "freq_dificuldade_concentracao_2sem",
    #"SAD_A": "k6_freq_tristeza_profunda", #infelizmente não tem 2025. PHQ82_A equivalente?
    "LONELY_A": "freq_solitário", #não está no paper mas deveria
    #"EFFORT_A": "k6_freq_esforco_extremo", não está no paper. Não achei equivalente
    #"PHQCAT_A": "escala_sintomas_depressão", #pode enviesar muito!
    
    "COGFRQDFF_A": "frequencia_falha_memoria", 
    "COGTYPEDFF_A": "tipo_dificuldade_cognitiva",
    "COGMEMDFF_A": "qtd_dificuldade_cognitiva",

    "ANXEV_A": "historico_transtorno_ansiedade",
    #"ANXMED_A": "med_ansiedade", #pode enviesar muito!
    "ANXFREQ_A": "freq_ansiedade",
    "ANXLEVEL_A": "intensidade_ansiedade",

    "GAD71_A": "freq_ansiedade_2sem",
    #"NERVOUS_A": "k6_freq_nervosismo",
    "GAD72_A": "freq_preocupacao_descontrolada_2sem",
    "GAD73_A": "freq_preocupacao_desnecessaria_2sem",
    "GAD74_A": "freq_dificuldade_descansar_2sem",
    "GAD75_A": "freq_agitacao_inquieta", # não tinha no paper mas é de ansiedade

    #"GADCAT_A": "escala_sintomas_ansiedade", #pode enviesar muito!
    #"K6SPD_A": "sofrimento_psicologico_k6", #não tem no paper nem 2025. Pode enviesar?
    ##############################################################################################################
    
    # Segurança alimentar: INFELIZMENTE NÃO TEM EM 2025
    #"FDSLESS_A": "comeu_menos_que_devia", 
    #"FDSWEIGHT_A": "perdeu_peso_restricao_alimentar", 
    #"FDSSKIP_A": "diminuiu_ou_pulou_refeicoes",
    #"FDSHUNGRY_A": "passou_fome_por_falta_dinheiro",
    
    #################################################################################################################
    
    #Dificuldade de andar
    "NOEQWLK13M_A": "andar_dificuldade_500m_sem_aparelho",
    "NOEQSTEPS_A": "andar_dificuldade_degraus_sem_aparelho",
    "EQWLK13M_A": "andar_dificuldade_500m_com_aparelho",
    "EQSTEPS_A": "andar_dificuldade_degraus_com_aparelho",
    "PERASST_A": "andar_necessita_ajuda_outra_pessoa",
    
    #####################################################################
    
    #
    
    "SUPPORT_A": "suporte_social", #não tem no artigo mas deveria

    #SCONNECT1_A    Number of times you get together with people you are close to 
    #SCONNECT2_A    Number of times you talk to people on telephone or by video 
    #SCONNECT3_A    Number of times you attend religious services 
    #SCONNECT4_A    Number of times you attend meetings of clubs or organizations 
    #PARENTCON1_A   How often receive social/emotional support parenting 
    #PARENTCON2_A   How difficult are day-to-day demands parenting 
    #PARENTCON3_A   How often do you talk, chat, or connect with other parents 
    
    #################################################################################################
    #Uso de cigarro

    "SMKNOW_A": "fuma_cigarro_atualmente",
    "CIGNOW_A": "qtd_cigarros_dia",
    "SMK30D_A": "dias_fumados_mes",
    "CIG30D_A": "qtd_cigarro_mes",
    "ECIGNOW_A": "usa_cigarro_eletronico_atualmente",
    "SMKEV_A": "ja_fumante", # na verdade foi perguntado já fumou 100 cigarros na vida. Isso categoriza como fumante? 

    
    
}

colunas_para_ler = list(MAPA_VARIAVEIS_PAPER.keys())
print(f"Colunas a serem carregadas: {len(colunas_para_ler)}")

# 2. Leitura otimizada direto do CSV
df_bruto = pd.read_csv("dados/adult25.csv", usecols=colunas_para_ler, low_memory=False)
print(f"Base carregada: {df_bruto.shape[0]:,} linhas e {df_bruto.shape[1]} colunas.")

# 3. Construção do Desfecho Clínico (target_medicacao)
# 1 = Sim, 2 = Não, outros = Indeterminado
#cond_sim = (df_bruto["DEPMED_A"] == 1) | (df_bruto["ANXMED_A"] == 1)
#cond_nao = (df_bruto["DEPMED_A"] == 2) & (df_bruto["ANXMED_A"] == 2)
#df_bruto["target_medicacao"] = np.where(cond_sim, 1, np.where(cond_nao, 0, np.nan))

# 4. Aplicação dos Nomes Padronizados em Português
df_limpo = df_bruto.rename(columns=MAPA_VARIAVEIS_PAPER)

# 5. Exportação para CSV limpo
nome_arquivo_saida = "dado_ingerido/adult25_limpo_selecionadas.csv"
df_limpo.to_csv(nome_arquivo_saida, index=False)

print(f"\nBase limpa exportada como '{nome_arquivo_saida}' com sucesso!")
print(f"Dimensões finais: {df_limpo.shape[0]:,} participantes e {df_limpo.shape[1]} atributos.")
print("\nDistribuição da variável dependente (target_medicacao):")
#print(df_limpo["target_medicacao"].value_counts(dropna=False, normalize=True).round(4) * 100)

Colunas a serem carregadas: 100
Base carregada: 24,215 linhas e 100 colunas.

Base limpa exportada como 'dado_ingerido/adult25_limpo_selecionadas.csv' com sucesso!
Dimensões finais: 24,215 participantes e 100 atributos.

Distribuição da variável dependente (target_medicacao):


In [74]:
df_limpo

,categoria_razao_pobreza,qtd_anos_estados_unidos,estado_civil,cojuge_trabalha,sexo_conjuge,categoria_imc,peso_libras,visitas_pronto_atendimento_12m,visitas_emergencia_hospitalar_12m,escolaridade_maxima_familia,...,historico_diabetes,asma_ativa_atualmente,remedio_colesterol,historico_colesterol,historico_hipertensao,satisfacao_com_a_vida,autoavaliacao_saude_geral,peso_amostral,id_domicilio,razao_renda_pobreza
0,6,NaN,1,2.0,1.0,3,171,0,0,4,...,2,NaN,NaN,2,2,2,3,10636.862,H012128,1.58
1,12,NaN,5,NaN,NaN,2,140,0,0,3,...,1,NaN,2.0,1,1,1,4,2996.860,H000617,4.32
2,1,NaN,7,NaN,NaN,3,175,0,0,2,...,2,NaN,NaN,2,2,2,3,29032.445,H033243,0.18
3,14,NaN,1,2.0,2.0,3,215,0,0,5,...,1,NaN,1.0,1,1,1,4,7875.107,H049140,6.32
4,13,NaN,7,NaN,NaN,4,198,0,0,8,...,2,NaN,1.0,1,1,2,2,4824.620,H012797,4.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24210,14,NaN,1,1.0,1.0,4,200,0,0,9,...,2,NaN,NaN,2,1,1,3,4869.749,H014073,6.18
24211,4,NaN,5,NaN,NaN,4,285,0,2,5,...,2,NaN,NaN,2,2,2,2,3350.243,H029805,1.21
24212,12,NaN,4,NaN,NaN,4,210,0,2,4,...,2,NaN,NaN,2,1,1,5,2374.594,H003491,4.32
24213,9,NaN,1,2.0,1.0,3,165,0,0,5,...,1,NaN,1.0,1,2,1,2,2400.569,H001251,2.63


Essas são as variáveis de características demográficas, gênero e suporte familiar


A variável AGE65 foi removida por apresentar respostas registradas em apenas 107 dos 24.215 registros. Ela se aplica somente a casos em que a idade numérica não foi informada, por isso sua baixa cobertura não é suficiente para o treinamento do modelo.

In [75]:
ID_DOMICILIO = "ID_DOMICILIO"
PESO_AMOSTRAL = "PESO_AMOSTRAL"
SEXO = "SEXO"
CLASSIFICACAO_URBANO_RURAL = "CLASSIFICACAO_URBANO_RURAL"
REGIAO_GEOGRAFICA = "REGIAO_GEOGRAFICA"
FAIXA_65_MAIS = "FAIXA_65_MAIS"

ESTADO_CIVIL_DECLARADO= "ESTADO_CIVIL_DECLARADO"
ESTADO_CIVIL= "ESTADO_CIVIL"
STATUS_PARENTAL= "STATUS_PARENTAL"
SEXO_CONJUGE= "SEXO_CONJUGE"
SEPARACAO_LEGAL_CONJUGAL= "SEPARACAO_LEGAL_CONJUGAL"
COJUGE_TRABALHA= "COJUGE_TRABALHA"

FLG_IDOSO_FAMILIA= "FLG_IDOSO_FAMILIA"
QTD_ADULTOS_FAMILIA= "QTD_ADULTOS_FAMILIA"
QTD_CRIANÇAS_FAMILIA= "QTD_CRIANÇAS_FAMILIA"
ESCOLARIDADE_MAXIMA_FAMILIA= "ESCOLARIDADE_MAXIMA_FAMILIA"
TEMPO_MORADIA_ANOS= "TEMPO_MORADIA_ANOS"
TIPO_POSSE_IMOVEL= "TIPO_POSSE_IMOVEL"

SUPORTE_SOCIAL= "SUPORTE_SOCIAL"

MAPA_VARIAVEIS_PAPER = {
  # Controles amostrais
    "HHX": ID_DOMICILIO,
    "WTFA_A": PESO_AMOSTRAL,
    #dados pessoais
    "SEX_A": SEXO,
    "URBRRL23": CLASSIFICACAO_URBANO_RURAL,
    "REGION": REGIAO_GEOGRAFICA,
    #"AGE65": FAIXA_65_MAIS,
    #dados família
    "MARITAL_A": ESTADO_CIVIL_DECLARADO,
    "MARSTAT_A": ESTADO_CIVIL,
    "PARSTAT_A": STATUS_PARENTAL, 
    "SPOUSESEX_A": SEXO_CONJUGE,
    "SPOUSEP_A": SEPARACAO_LEGAL_CONJUGAL,
    "SPOUSWRK_A": COJUGE_TRABALHA,
    "OVER65FLG_A": FLG_IDOSO_FAMILIA,
    "PCNTADLT_A": QTD_ADULTOS_FAMILIA,
    "PCNTKIDS_A": QTD_CRIANÇAS_FAMILIA,
    "MAXEDUCP_A": ESCOLARIDADE_MAXIMA_FAMILIA,
    
    #moradia
    "HOUYRSLIV_A": TEMPO_MORADIA_ANOS,
    #"HOUSECOST_A": "dificuldade_custo_moradia", #não estava no paper e não tem em 2025
    "HOUTENURE_A": TIPO_POSSE_IMOVEL, 
   

    "SUPPORT_A": SUPORTE_SOCIAL, #não tem no artigo mas deveria
}

Como cerca de 80% a 85% dos entrevistados no inquérito nasceram nos EUA, mais de 80% da base fica automaticamente sem resposta nessa variável, gerando essa frequência elevada de missingness que observamos.
Por isso substituimos pela variável NATUSBORN_A que pergunta a naturalidade estadunidense do entrevistado.

In [76]:
QTD_ANOS_ESTADOS_UNIDOS= "QTD_ANOS_ESTADOS_UNIDOS"
NATURAL_ESTADOS_UNIDOS = "NATURAL_ESTADOS_UNIDOS"

MAPA_VARIAVEIS_PAPER.update(
    {
    "NATUSBORN_A": NATURAL_ESTADOS_UNIDOS,
    #"YRSINUS_A": QTD_ANOS_ESTADOS_UNIDOS,
    }
)

Essas são as variavais de saúde geral

In [77]:
AUTOAVALIACAO_SAUDE_GERAL = "AUTOAVALIACAO_SAUDE_GERAL"
CATEGORIA_IMC = "CATEGORIA_IMC"
PESO_LIBRAS = "PESO_LIBRAS"
SATISFACAO_COM_A_VIDA = "SATISFACAO_COM_A_VIDA"
LIMITACAO_TRABALHO_POR_SAUDE ="LIMITACAO_TRABALHO_POR_SAUDE"

HISTORICO_HIPERTENSAO ="HISTORICO_HIPERTENSAO"

INTERROMPEU_INSULINA_PRM_ANO = "INTERROMPEU_INSULINA_PRM_ANO"
HISTORICO_ARTRITE = "HISTORICO_ARTRITE"

CONCUSSAO_SINTOMAS_POS_TRAUMA ="CONCUSSAO_SINTOMAS_POS_TRAUMA"



MAPA_VARIAVEIS_PAPER.update(
   {
        #saude geral
        "PHSTAT_A": AUTOAVALIACAO_SAUDE_GERAL, 
        "BMICAT_A": CATEGORIA_IMC,
        "WEIGHTLBTC_A": PESO_LIBRAS,
        "LSATIS4_A": SATISFACAO_COM_A_VIDA,
        "SOCWRKLIM_A": LIMITACAO_TRABALHO_POR_SAUDE,
        
        
        "HYPEV_A": HISTORICO_HIPERTENSAO,

        "ARTHEV_A": HISTORICO_ARTRITE,

         
        
        "TBIHLSBMC_A": CONCUSSAO_SINTOMAS_POS_TRAUMA, #não tem no & artigo não tem em 2025
        #"INJFALL_A": "lesao_decorrente_queda",#não tem no artigo & não tem em 2025
        #"INJFALLHOM_A": "queda_ocorrida_em_casa", #não tem no artigo & não tem em 2025

        #"REPWRKCAUS_A": "ler_dort_causada_no_trabalho", não tem em 2025
        #"REPSTRAIN_A": "lesao_por_esforco_repetitivo", não tem em 2025
        #"REPLIMIT_A": "limitacao_por_ler_dort", não tem em 2025

        #vacinação




        #"INJWRKDYTC_A": "problema_saude_dias_sem_trabalhar"

        

   } 
)


Mantemos apenas CHLEV_A porque ela representa o histórico de colesterol elevado — o conceito que queremos analisar — e é mapeada para HISTORICO_COLESTEROL no notebook. CHLMED_A, por sua vez, registra o uso de medicamentos para colesterol: é uma variável de tratamento, não uma medida equivalente do histórico da condição. Por isso, ela permanece fora do mapeamento. A decisão também evita confundir o histórico de colesterol com o tratamento recebido.

In [78]:
HISTORICO_COLESTEROL ="HISTORICO_COLESTEROL"
REMEDIO_COLESTEROL= "REMEDIO_COLESTEROL"


MAPA_VARIAVEIS_PAPER.update(
   {

        "CHLEV_A": HISTORICO_COLESTEROL,
        #"CHLMED_A": REMEDIO_COLESTEROL,

   } 
)

Substituímos apenas por DIBEV_A porque ela mede o histórico de diabetes — o conceito que queremos representar — e, conforme os dados informados, não tem valores ausentes. No notebook, ela é mapeada para HISTORICO_DIABETES (02_preparacao_e_limpeza_2025_mdd.ipynb).

As outras variáveis não são alternativas equivalentes: DIBPILL_A registra uso de comprimidos para diabetes e tem 76,85% de valores ausentes; DIBINSSTYR_A trata de interrupção de insulina no último ano e tem 99,897% ausentes. Além da baixa cobertura, elas descrevem tratamentos ou comportamentos específicos, não se a pessoa tem histórico de diabetes. Os ausentes podem decorrer da aplicabilidade ou do fluxo das perguntas; não devem ser interpretados automaticamente como “não”.

Assim, usar apenas DIBEV_A preserva a cobertura da amostra e mantém a variável alinhada ao conceito analisado, sem confundir diagnóstico/histórico com tratamento.

In [79]:
HISTORICO_DIABETES = "HISTORICO_DIABETES"
INTERROMPEU_INSULINA_PRM_ANO = "INTERROMPEU_INSULINA_PRM_ANO"
MED_DIABETES_ORAL = "MED_DIABETES_ORAL"


MAPA_VARIAVEIS_PAPER.update(
   {
       
      "DIBEV_A": HISTORICO_DIABETES,
      #"DIBPILL_A": MED_DIABETES_ORAL, # no paper usou dibpill mas tem vários outros tratamentos
      #"DIBINSSTYR_A": INTERROMPEU_INSULINA_PRM_ANO,

   } 
)

### Otimização do Bloco de Mobilidade e Dificuldades Motoras

**Problema Identificado:**
As variáveis detalhadas de caminhada e uso de escadas (`NOEQWLK13M_A`, `EQWLK13M_A`, `NOEQSTEPS_A` e `EQSTEPS_A`) apresentavam **taxas de *missingness* elevadas (entre 90.4% e 94.1%)** na amostra de 24.215 participantes[cite: 5]:

* **Caminhar 500m sem aparelho (`NOEQWLK13M_A`):** 94.08% de nulos
* **Caminhar 500m com aparelho (`EQWLK13M_A`):** 94.23% de nulos
* **Andar dificuldade degraus sem aparelho (`NOEQSTEPS_A`):** 90.51% de nulos
* **Andar dificuldade degraus com aparelho (`EQSTEPS_A`):** 93.11% de nulos

Essa escassez de dados ocorre devido aos saltos condicionais (*skip patterns*) do questionário do CDC, que dividem a população entre quem usa e não usa equipamentos de auxílio. Aplicar imputação estatística sobre esses campos geraria ruído severo e multicolinearidade nos modelos.

**Solução Aplicada:**
1. **Substituição por Variável Geral (`DIFF_A`):** Eliminamos as 4 colunas detalhadas e adotamos a variável nativa de triagem **`DIFF_A`** (`DIFICULDADE_LOCOMOCAO_GERAL`), aplicada diretamente a toda a população do inquérito.
2. **Integridade da Base:** A variável `DIFF_A` possui **0% de *missingness* estrutural**, garantindo cobertura completa para todos os 24.215 participantes.
3. **Padronização e Escala:** A variável foi recodificada para a escala de $0$ a $3$ ($0 = \text{Sem dificuldade}$ até $3 = \text{Incapaz}$), mantendo o sinal clínico de limitação motora de forma coesa para os algoritmos de Machine Learning.

In [80]:
DIFICULDADE_LOCOMOCAO = "DIFICULDADE_LOCOMOCAO"

ANDAR_DIFICULDADE_500M_SEM_APARELHO = "ANDAR_DIFICULDADE_500M_SEM_APARELHO"
ANDAR_DIFICULDADE_DEGRAUS_SEM_APARELHO = "ANDAR_DIFICULDADE_DEGRAUS_SEM_APARELHO"
ANDAR_DIFICULDADE_500M_COM_APARELHO = "ANDAR_DIFICULDADE_500M_COM_APARELHO"
ANDAR_DIFICULDADE_DEGRAUS_COM_APARELHO = "ANDAR_DIFICULDADE_DEGRAUS_COM_APARELHO"
ANDAR_NECESSITA_AJUDA_OUTRA_PESSOA = "ANDAR_NECESSITA_AJUDA_OUTRA_PESSOA"


MAPA_VARIAVEIS_PAPER.update({
    "DIFF_A": DIFICULDADE_LOCOMOCAO

    #Difivuldade de andar
    #"NOEQWLK13M_A": ANDAR_DIFICULDADE_500M_SEM_APARELHO,
    #"NOEQSTEPS_A": ANDAR_DIFICULDADE_DEGRAUS_SEM_APARELHO,
    #"EQWLK13M_A": ANDAR_DIFICULDADE_500M_COM_APARELHO,
    #"EQSTEPS_A": ANDAR_DIFICULDADE_DEGRAUS_COM_APARELHO,
    #"PERASST_A": ANDAR_NECESSITA_AJUDA_OUTRA_PESSOA,
})



A variável ASTILL_A diz se a pessoa tem histórico de Asma e ainda tem asma, tendo frequencia de 3.491 registros de 24.215, 14.42%.
Fica a dúvida: utilizamos ASEV_A que diz se a pessoa teve asma ou não ou usamos a ASTILL_A e botamos o valor "não" para todos os registros vazios?
Pessoas que tem asma hoje são 2062 => 8.52%

In [81]:
HISTORICO_ASMA = "HISTORICO_ASMA"
ASMA_ATIVA_ATUALMENTE ="ASMA_ATIVA_ATUALMENTE"

MAPA_VARIAVEIS_PAPER.update(
   {
      "ASTILL_A": ASMA_ATIVA_ATUALMENTE,
      #"ASEV_A": HISTORICO_ASMA
   }
) 

A variável PAIAPG3M_A que aparece no paper pergunta se a pessoa teve dores abdominais. Apesar de ter relativamente poucos valores nulos: 8.981 de 24215 (37%), 92% dos 15.234 restantes responderam "Not at all" ou "A little" para o nível de dor abdominal sentida.

As respostas:
"Somewhere in between a little and a lot": 529
"A lot" : 606

São apenas 2,1% e 2,5% respectivamente. 


Portanto a variável PAIAPG3M_A será substituida por PAIFRQ3M_A: A frequencia que a pessoa esteve com dor nos ultimos 3 meses.

PAIAPG3M_A é diluida em diversas perguntas que especificam a parte do corpo que a pessoa sente dor cronica. 

PAIBACK3M_A: Coluna
PAIULMB3M_A: Mãos
PAILLMB3M_A: Cintura
PAIHDFC3M_A: Cabeça
PAIAPG3M_A:  Abdominal
PAITOOTH3M_A:Dente ou mandibula

Por isso a pouca quantidade de valores relevantes

In [82]:
DOR_CRONICA = "DOR_CRONICA"


MAPA_VARIAVEIS_PAPER.update(
    {
        "PAIFRQ3M_A": DOR_CRONICA
        
    }
)

As variáveis SHTFLU12M_A, SHTFLUY_A tratam de vacinação de gripe. 
SHTFLU12M_A: Se a pessoa vacinou-se nos ultimos 12 meses. 
SHTFLUY_A: Se ela respondeu sim na resposta acima, apenas tem valores 2024 e 2025. Não tem valores de outros anos como inicialmente foi pensado. 

Portanto SHTFLUY_A é redundante. É para ser usada com SHTFLUM_A para saber o mês e ano no qual a pessoa tomou a vacina da gripe neste ano 


In [83]:
##### ANÁLISE PARA MANTER SHTFLU12M_A
ANO_VACINA_GRIPE = "ANO_VACINA_GRIPE"
VACINA_GRIPE_ULTIMOS_12M = "VACINA_GRIPE_ULTIMOS_12M"
MAPA_VACINACAO = {
    "SHTFLUY_A": ANO_VACINA_GRIPE,
    "SHTFLU12M_A": VACINA_GRIPE_ULTIMOS_12M,
    
   
}

colunas_para_ler = list(MAPA_VACINACAO.keys())

df_bruto = pd.read_csv("dados/adult25.csv", usecols=colunas_para_ler, low_memory=False)

df_vacina = df_bruto.rename(columns=MAPA_VACINACAO)

df_vacina

pd.crosstab(
df_vacina["VACINA_GRIPE_ULTIMOS_12M"],
df_vacina["ANO_VACINA_GRIPE"],
dropna=False)

ANO_VACINA_GRIPE,2024.0,2025.0,9997.0,9999.0,NaN
VACINA_GRIPE_ULTIMOS_12M,,,,,
1,8480,2532,1,68,16
2,0,0,0,0,12675
7,0,0,0,0,37
8,0,0,0,0,346
9,0,0,0,0,60


In [84]:
DOR_CRONICA = "DOR_CRONICA"


MAPA_VARIAVEIS_PAPER.update(
    {
        "SHTFLU12M_A": VACINA_GRIPE_ULTIMOS_12M
        
    }
)

In [85]:
##### ANÁLISE PARA MANTER QTD_VACINA_PNEUMONIA
QTD_VACINA_PNEUMONIA = "QTD_VACINA_PNEUMONIA"
JA_TOMOU_VACINA_PNEUMONIA = "JA_TOMOU_VACINA_PNEUMONIA"

MAPA_VACINACAO = {

    
    "SHTPNEUNB_A": QTD_VACINA_PNEUMONIA,
    "SHTPNUEV_A": JA_TOMOU_VACINA_PNEUMONIA,#
}

colunas_para_ler = list(MAPA_VACINACAO.keys())

df_bruto = pd.read_csv("dados/adult25.csv", usecols=colunas_para_ler, low_memory=False)

df_vacina = df_bruto.rename(columns=MAPA_VACINACAO)

df_vacina

pd.crosstab(
df_vacina[QTD_VACINA_PNEUMONIA],
df_vacina[JA_TOMOU_VACINA_PNEUMONIA],
dropna=False)

JA_TOMOU_VACINA_PNEUMONIA,1,2,7,8,9
QTD_VACINA_PNEUMONIA,,,,,
1.0,3255,0,0,0,0
2.0,2512,0,0,0,0
3.0,1291,0,0,0,0
7.0,6,0,0,0,0
9.0,461,0,0,0,0
NaN,0,15635,28,355,672


In [86]:
ANO_VACINA_GRIPE = "ANO_VACINA_GRIPE"
VACINA_GRIPE_ULTIMOS_12M = "VACINA_GRIPE_ULTIMOS_12M"
QTD_VACINA_PNEUMONIA = "QTD_VACINA_PNEUMONIA"
VACINA_TETANUS_10A = "VACINA_TETANUS_10A"


MAPA_VARIAVEIS_PAPER.update(
    {
        
        "SHTFLUY_A": ANO_VACINA_GRIPE,
        #"SHTFLU12M_A": VACINA_GRIPE_ULTIMOS_12M,

        "SHTPNEUNB_A": QTD_VACINA_PNEUMONIA,
        "SHTTETANUS_A": VACINA_TETANUS_10A,#
        
    }
)








In [87]:
#df_vacina[ANO_VACINA_GRIPE].value_counts().sort_index()
df_vacina.head()

,QTD_VACINA_PNEUMONIA,JA_TOMOU_VACINA_PNEUMONIA
0,2.0,1
1,NaN,9
2,NaN,2
3,1.0,1
4,NaN,2


In [88]:

#pd.crosstab(
#    df_vacina["VACINA_GRIPE_ULTIMOS_12M"],
#    df_vacina["ANO_VACINA_GRIPE"].isna()
#)

As variáveis: CERVIAGETC_A, PANCRAGETC_A, ESOPHAGETC_A, LUNGAGETC_A, LIVERAGETC_A, SKNDKAGETC_A, SKNDKAGETC_A, THYROAGETC_A, STOMAAGETC_A,    foram retiradas devido as sua grande quantidade de valores nulos, sendo impossível a sua utilização em treinamento pois não pode ser preenchida com valores Moda.

Dos 24.215 mil registros as variáveis apresentaram-se apenas:

CERVIAGETC_A : 13
PANCRAGETC_A : 13
ESOPHAGETC_A : 17
LUNGAGETC_A  : 120
LIVERAGETC_A : 22
SKNDKAGETC_A : 114
THYROAGETC_A : 108
STOMAAGETC_A : 20

Essas variáveis foram substituidas pela variável CANEV_A : "já teve cancer?"
CANEV_A foi preenchida em todos os registros, com a seguinte frequencia de respostas abaixo

Yes 3378  13.95%
No  20804 85.91%


In [89]:
#IDADE_CANCER_COLO_UTERO = "IDADE_CANCER_COLO_UTERO"
#IDADE_CANCER_PANCREAS ="IDADE_CANCER_PANCREAS"
#IDADE_CANCER_ESOFAGO = "IDADE_CANCER_ESOFAGO"
#IDADE_CANCER_PULMAO = "IDADE_CANCER_PULMAO"
#IDADE_CANCER_FIGADO = "IDADE_CANCER_FIGADO"
#IDADE_CANCER_PELE_INDETERMINADO = "IDADE_CANCER_PELE_INDETERMINADO"
#IDADE_CANCER_TIREOIDE = "IDADE_CANCER_TIREOIDE"
#IDADE_CANCER_ESTOMAGO = "IDADE_CANCER_ESTOMAGO"

TEVE_CANCER = "TEVE_CANCER"


MAPA_VARIAVEIS_PAPER.update(
   {
        #câncer

        #"CERVIAGETC_A": IDADE_CANCER_COLO_UTERO,
        #"PANCRAGETC_A": IDADE_CANCER_PANCREAS, #
        #"ESOPHAGETC_A": IDADE_CANCER_ESOFAGO, #
        #"LUNGAGETC_A": IDADE_CANCER_PULMAO,
        #"LIVERAGETC_A": IDADE_CANCER_FIGADO,
        #"SKNDKAGETC_A": IDADE_CANCER_PELE_INDETERMINADO,
        #"THYROAGETC_A": IDADE_CANCER_TIREOIDE,
        #"STOMAAGETC_A": IDADE_CANCER_ESTOMAGO,
         "CANEV_A": TEVE_CANCER
   } 
)

As variáveis WELLNESS_A e WELLVIS_A foram substituídas por LASTDR_A, que registra há quanto tempo o adulto consultou um médico ou profissional de saúde. LASTDR_A se aplica a todos os adultos e possui respostas para os 24.215 registros, enquanto WELLNESS_A se aplica apenas a quem consultou um médico e WELLVIS_A a um grupo ainda mais restrito, resultando em muitos valores ausentes. Assim, LASTDR_A oferece uma medida mais abrangente e com maior cobertura sobre a utilização de serviços de saúde.

In [90]:
TEMPO_ULTIMO_CHECKUP_GERAL="TEMPO_ULTIMO_CHECKUP_GERAL"
ULTIMA_VISITA_FOI_CHECKUP="ULTIMA_VISITA_FOI_CHECKUP"
ULTIMA_VISITA_MEDICO = "ULTIMA_VISITA_MEDICO"

VISITAS_PRONTO_ATENDIMENTO_12M="VISITAS_PRONTO_ATENDIMENTO_12M"
VISITAS_EMERGENCIA_HOSPITALAR_12M="VISITAS_EMERGENCIA_HOSPITALAR_12M"
VISITAS_EXAME_VISTA_12M="VISITAS_EXAME_VISTA_12M"

MAPA_VARIAVEIS_PAPER.update(
    {
   
        #plano de saude
        "LASTDR_A": ULTIMA_VISITA_MEDICO,
    
        "URGCC12MTC_A": VISITAS_PRONTO_ATENDIMENTO_12M,
        "EMERG12MTC_A": VISITAS_EMERGENCIA_HOSPITALAR_12M,
        "EYEEX12M_A": VISITAS_EXAME_VISTA_12M,
        #"WELLVIS_A": TEMPO_ULTIMO_CHECKUP_GERAL,
        #"WELLNESS_A": ULTIMA_VISITA_FOI_CHECKUP, #meio redundante

        #"INJSAWDOC_A": "atendimento_medico_por_lesao", não tem e, 2025
    }
)


A variável FLUNCH12M1_A foi removida por ter respostas registradas em apenas 4.712 dos 24.215 registros. Ela se aplica somente a adultos que vivem em famílias com crianças de 5 a 17 anos, o que limita sua cobertura na base e sua utilidade para o treinamento do modelo.

In [91]:
MERENDA_ESCOLAR_GRATUITA = "MERENDA_ESCOLAR_GRATUITA"
CATEGORIA_RAZAO_POBREZA= "CATEGORIA_RAZAO_POBREZA"
RAZAO_RENDA_POBREZA= "RAZAO_RENDA_POBREZA"
RENDA_INVESTIMENTOS="RENDA_INVESTIMENTOS"


MAPA_VARIAVEIS_PAPER.update(
    {
        #economicos
        #"FLUNCH12M1_A": MERENDA_ESCOLAR_GRATUITA,
        "RATCAT_A": CATEGORIA_RAZAO_POBREZA,
        "POVRATTC_A": RAZAO_RENDA_POBREZA,
        "INCINTER_A":RENDA_INVESTIMENTOS,
        
        
        
        #"RSNHICOST_A": SEM_PLANO_POR_CUSTO_INACESSIVEL,

        #plano de saude

    }
)

### Seleção das variáveis de cobertura de saúde

**MEDICARE_A** e **PRIVATE_A** foram mantidas como variáveis de contexto, pois a cobertura de saúde pode influenciar o acesso a diagnóstico, tratamento e acompanhamento. **PAYWORRY_A** também foi mantida como indicador de estresse financeiro relacionado a despesas médicas.

As demais variáveis foram excluídas por medirem uso de serviços ou detalhes específicos de planos que não são centrais à análise. Algumas se aplicam apenas a grupos restritos; portanto, seus valores ausentes podem indicar que a pergunta não se aplicava ao participante, e não necessariamente que houve perda de resposta.

| Variável | Decisão | Justificativa resumida |
|---|---|---|
| **VAHOSP_A** | Excluir | Mede atendimento recebido ou pago pela VA nos últimos 12 meses, não uma condição de saúde. |
| **PLN1PAY2_A** | Excluir | Informa se empregador ou sindicato paga o primeiro plano privado. É um detalhe sobre o pagamento do plano; **PRIVATE_A** foi mantida como indicador mais geral de cobertura privada. |
| **PRDEDUC1_A** | Excluir | Refere-se à franquia anual do primeiro plano privado, um detalhe do plano e não uma medida de saúde. |
| **PRDEDUC2_A** | Excluir | Refere-se à franquia de um segundo plano privado; apenas 317 participantes têm resposta registrada (1,31%). |
| **OPDEDUC_A** | Excluir | Refere-se à franquia de um plano estadual; apenas 116 participantes têm resposta registrada (0,48%). |
| **HINOTMYR_A** | Excluir | Registra meses sem cobertura em um grupo específico e, conforme o codebook, não deve ser usada para estimar quem tem cobertura. |
| **PAYNOBLLNW_A** | Excluir | Apenas 2.420 dos 24.215 registros têm respostas codificadas. A pergunta se aplica a um grupo específico, o que limita sua cobertura para o treinamento do modelo. |
| **HISTOPJOB_A** | Excluir | Tem respostas registradas em apenas 829 dos 24.215 registros e se aplica somente a um grupo específico de adultos sem cobertura. |
| **RSNHICOST_A** | Excluir | Apenas 1.487 dos 24.215 registros têm resposta “Sim” ou “Não”. A variável se aplica somente a adultos sem cobertura de saúde. |

A seleção mantém variáveis gerais de cobertura e estresse financeiro relacionado às despesas médicas, enquanto exclui informações muito específicas ou com cobertura limitada na base. Nas perguntas de universo restrito, os valores ausentes não devem ser interpretados automaticamente como respostas negativas.


In [92]:
PLANO_PUBLICO = "PLANO_PUBLICO"
PLANO_PRIVADO = "PLANO_PRIVADO"

PLANO_ESTADUAL_DEDUTIVEL="PLANO_ESTADUAL_DEDUTIVEL"
PLANO_PRIVADO_DEDUTIVEL="PLANO_PRIVADO_DEDUTIVEL"
PLANO_PRIVADO_DEDUTIVEL2="PLANO_PRIVADO_DEDUTIVEL2"
PLANO_MILITAR_SAUDE="PLANO_MILITAR_SAUDE"
PLANO_PATRAO_SINDICATO_PAGAM="PLANO_PATRAO_SINDICATO_PAGAM"
PLANO_MESES_SEM="PLANO_MESES_SEM"

SEM_PLANO_POR_CUSTO_INACESSIVEL="SEM_PLANO_POR_CUSTO_INACESSIVEL"
PERDEU_PLANO_DEMITIDO_OU_TROCOU_EMPREGO="PERDEU_PLANO_DEMITIDO_OU_TROCOU_EMPREGO"

ESTRESSE_FINANCEIRO_MEDICO="ESTRESSE_FINANCEIRO_MEDICO"

MAPA_VARIAVEIS_PAPER.update(
    {
        #plano de saude
        "MEDICARE_A" : PLANO_PUBLICO,
        "PRIVATE_A": PLANO_PRIVADO,
        "PAYWORRY_A": ESTRESSE_FINANCEIRO_MEDICO,
        
        #"OPDEDUC_A": PLANO_ESTADUAL_DEDUTIVEL,

        #"OPDEDUC_A": PLANO_ESTADUAL_DEDUTIVEL,
        #"PRDEDUC1_A": PLANO_PRIVADO_DEDUTIVEL,
        #"PRDEDUC2_A": PLANO_PRIVADO_DEDUTIVEL2,
        #"VAHOSP_A": PLANO_MILITAR_SAUDE,
        #"HISTOPJOB_A":PERDEU_PLANO_DEMITIDO_OU_TROCOU_EMPREGO,
        #"PLN1PAY2_A": PLANO_PATRAO_SINDICATO_PAGAM,
        #"HINOTMYR_A": PLANO_MESES_SEM, #
        #"PAYNOBLLNW_A": DIVIDAS_MEDICAS_EM_ABERTO,
        
        #"PLN1PAY2_A": PLANO_PATRAO_SINDICATO_PAGAM,
        #"PRDEDUC1_A": PLANO_PRIVADO_DEDUTIVEL1,
        #"PRDEDUC2_A": PLANO_PRIVADO_DEDUTIVEL2,
        
    }
)

Essas são condições de estilo de vida e padrões comportamentais

In [93]:
FUMA_CIGARRO_ATUALMENTE= "FUMA_CIGARRO_ATUALMENTE"
QTD_CIGARROS_DIA= "QTD_CIGARROS_DIA"
DIAS_FUMADOS_MES= "DIAS_FUMADOS_MES"
QTD_CIGARRO_MES= "QTD_CIGARRO_MES"
USA_CIGARRO_ELETRONICO_ATUALMENTE= "USA_CIGARRO_ELETRONICO_ATUALMENTE"
JA_FUMANTE= "JA_FUMANTE"

MAPA_VARIAVEIS_PAPER.update(
    {
        "SMKNOW_A": FUMA_CIGARRO_ATUALMENTE,
        "CIGNOW_A": QTD_CIGARROS_DIA,
        "SMK30D_A": DIAS_FUMADOS_MES,
        "CIG30D_A": QTD_CIGARRO_MES,
        "ECIGNOW_A": USA_CIGARRO_ELETRONICO_ATUALMENTE,
        "SMKEV_A": JA_FUMANTE, # na verdade foi perguntado já fumou 100 cigarros na vida. Isso categoriza como fumante? 
    }
)


Essas são condições relacionadas a ansiedade e depressão

In [94]:
HISTORICO_DEPRESSAO= "HISTORICO_DEPRESSAO"
REMEDIO_DEPRESSAO= "REMEDIO_DEPRESSAO"
FREQ_DEPRESSAO= "FREQ_DEPRESSAO"
INTENSIDADE_DEPRESSAO= "INTENSIDADE_DEPRESSAO"
FREQ_PERDA_INTERESSE_2SEM= "FREQ_PERDA_INTERESSE_2SEM"
FREQ_SENTIU_DEPRIMIDO_2SEM= "FREQ_SENTIU_DEPRIMIDO_2SEM"


FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM= "FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM"
FREQ_DIFICULDADE_DORMIR_2SEM= "FREQ_DIFICULDADE_DORMIR_2SEM"
FREQ_CANSASO_2SEM= "FREQ_CANSASO_2SEM"
FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM= "FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM"
FREQ_AUTOAVALIACAO_NEGTIVA_2SEM= "FREQ_AUTOAVALIACAO_NEGTIVA_2SEM"
FREQ_DIFICULDADE_CONCENTRACAO_2SEM= "FREQ_DIFICULDADE_CONCENTRACAO_2SEM"


FREQ_SOLITÁRIO= "FREQ_SOLITÁRIO"


FREQUENCIA_FALHA_MEMORIA= "FREQUENCIA_FALHA_MEMORIA"
TIPO_DIFICULDADE_COGNITIVA= "TIPO_DIFICULDADE_COGNITIVA"
QTD_DIFICULDADE_COGNITIVA= "QTD_DIFICULDADE_COGNITIVA"


HISTORICO_TRANSTORNO_ANSIEDADE= "HISTORICO_TRANSTORNO_ANSIEDADE"
MED_ANSIEDADE= "MED_ANSIEDADE"
FREQ_ANSIEDADE= "FREQ_ANSIEDADE"
INTENSIDADE_ANSIEDADE= "INTENSIDADE_ANSIEDADE"

FREQ_ANSIEDADE_2SEM= "FREQ_ANSIEDADE_2SEM"
FREQ_PREOCUPACAO_DESCONTROLADA_2SEM= "FREQ_PREOCUPACAO_DESCONTROLADA_2SEM"
FREQ_PREOCUPACAO_DESNECESSARIA_2SEM= "FREQ_PREOCUPACAO_DESNECESSARIA_2SEM"
FREQ_DIFICULDADE_DESCANSAR_2SEM= "FREQ_DIFICULDADE_DESCANSAR_2SEM"
FREQ_AGITACAO_INQUIETA= "FREQ_AGITACAO_INQUIETA"

MAPA_VARIAVEIS_PAPER.update(
    {
    #Ansiedade, Depressão e seus Sintomas ########################################################

    "DEPEV_A": HISTORICO_DEPRESSAO,
    "DEPMED_A": REMEDIO_DEPRESSAO, #pode enviesar muito!
    "DEPFREQ_A": FREQ_DEPRESSAO,
    "DEPLEVEL_A": INTENSIDADE_DEPRESSAO,
    "PHQ81_A": FREQ_PERDA_INTERESSE_2SEM,
    "PHQ82_A": FREQ_SENTIU_DEPRIMIDO_2SEM,
    #"HOPELESS_A": "k6_freq_desesperanca", #infelizmente não tem 2025. PHQ82_A equivalente?
    #"WORTHLESS_A": "k6_freq_desvalorizacao", #infelizmente não tem 2025. PHQ82_A equivalente?
    "GAD72_A": FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM,
    "PHQ83_A": FREQ_DIFICULDADE_DORMIR_2SEM,
    #"SLPFLL_A": "dificuldade_adormecer", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPSTY_A": "dificuldade_manter_sono", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPREST_A": "acorda_descansado", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    "PHQ84_A": FREQ_CANSASO_2SEM,
    #"FGEFRQTRD_A": "frequencia_fadiga_3m", #infelizmente não tem no paper e 2025. PHQ84_A equivalente?
    "PHQ85_A": FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM,
    "PHQ86_A": FREQ_AUTOAVALIACAO_NEGTIVA_2SEM, #auto estima, etc...
    "PHQ87_A": FREQ_DIFICULDADE_CONCENTRACAO_2SEM,
    #"SAD_A": "k6_freq_tristeza_profunda", #infelizmente não tem 2025. PHQ82_A equivalente?
    "LONELY_A": FREQ_SOLITÁRIO, #não está no paper mas deveria
    #"EFFORT_A": "k6_freq_esforco_extremo", não está no paper. Não achei equivalente
    #"PHQCAT_A": "escala_sintomas_depressão", #pode enviesar muito!
    
    "COGFRQDFF_A": FREQUENCIA_FALHA_MEMORIA, 
    "COGTYPEDFF_A": TIPO_DIFICULDADE_COGNITIVA,
    "COGMEMDFF_A": QTD_DIFICULDADE_COGNITIVA,

    "ANXEV_A": HISTORICO_TRANSTORNO_ANSIEDADE,
    "ANXMED_A": MED_ANSIEDADE, #pode enviesar muito!
    "ANXFREQ_A": FREQ_ANSIEDADE,
    "ANXLEVEL_A": INTENSIDADE_ANSIEDADE,

    "GAD71_A": FREQ_ANSIEDADE_2SEM,
    #"NERVOUS_A": "k6_freq_nervosismo",
    "GAD72_A": FREQ_PREOCUPACAO_DESCONTROLADA_2SEM,
    "GAD73_A": FREQ_PREOCUPACAO_DESNECESSARIA_2SEM,
    "GAD74_A": FREQ_DIFICULDADE_DESCANSAR_2SEM,
    "GAD75_A": FREQ_AGITACAO_INQUIETA, # não tinha no paper mas é de ansiedade

    #"GADCAT_A": "escala_sintomas_ansiedade", #pode enviesar muito!
    #"K6SPD_A": "sofrimento_psicologico_k6", #não tem no paper nem 2025. Pode enviesar?
    ##############################################################################################################
    
    # Segurança alimentar: INFELIZMENTE NÃO TEM EM 2025
    #"FDSLESS_A": "comeu_menos_que_devia", 
    #"FDSWEIGHT_A": "perdeu_peso_restricao_alimentar", 
    #"FDSSKIP_A": "diminuiu_ou_pulou_refeicoes",
    #"FDSHUNGRY_A": "passou_fome_por_falta_dinheiro",
    
    }
)




In [95]:
# ==============================================================================
# PIPELINE DE EXTRAÇÃO, HIGIENIZAÇÃO E SALVAMENTO DA BASE LIMPA (NHIS 2025)
# ==============================================================================
import numpy as np
import pandas as pd

YES_NO = {
    1 : "SIM",
    2 : "NÃO",
    7 : np.nan,
    8 : np.nan,
    9 : np.nan
}

QTD = {
    1: "1",
    2: "2",
    3: "3+",
    4: "INCERTO"
}

# 1. Dicionário Consolidado das Variáveis Selecionadas (101 Features + Controles)


colunas_para_ler = list(MAPA_VARIAVEIS_PAPER.keys())
print(f"Colunas a serem carregadas: {len(colunas_para_ler)}")

# 2. Leitura otimizada direto do CSV
df_bruto = pd.read_csv("dados/adult25.csv", usecols=colunas_para_ler, low_memory=False)
print(f"Base carregada: {df_bruto.shape[0]:,} linhas e {df_bruto.shape[1]} colunas.")

# 3. Construção do Desfecho Clínico (target_medicacao)
# 1 = Sim, 2 = Não, outros = Indeterminado
#cond_sim = (df_bruto["DEPMED_A"] == 1) | (df_bruto["ANXMED_A"] == 1)
#cond_nao = (df_bruto["DEPMED_A"] == 2) & (df_bruto["ANXMED_A"] == 2)
#df_bruto["target_medicacao"] = np.where(cond_sim, 1, np.where(cond_nao, 0, np.nan))

# 4. Aplicação dos Nomes Padronizados em Português
df_limpo = df_bruto.rename(columns=MAPA_VARIAVEIS_PAPER)

# 5. Exportação para CSV limpo
nome_arquivo_saida = "dado_ingerido/adult25_limpo_selecionadas.csv"
df_limpo.to_csv(nome_arquivo_saida, index=False)

print(f"\nBase limpa exportada como '{nome_arquivo_saida}' com sucesso!")
print(f"Dimensões finais: {df_limpo.shape[0]:,} participantes e {df_limpo.shape[1]} atributos.")
print("\nDistribuição da variável dependente (target_medicacao):")
#print(df_limpo["target_medicacao"].value_counts(dropna=False, normalize=True).round(4) * 100)

Colunas a serem carregadas: 77
Base carregada: 24,215 linhas e 77 colunas.

Base limpa exportada como 'dado_ingerido/adult25_limpo_selecionadas.csv' com sucesso!
Dimensões finais: 24,215 participantes e 77 atributos.

Distribuição da variável dependente (target_medicacao):


In [ ]:
# ==============================================================================
# HIGIENIZAÇÃO E CODIFICAÇÃO COMPLETA (93 FEATURES + TARGET)
# ==============================================================================


# 1. Higienização universal de sentinelas do CDC (7=Recusou, 8=Incerto, 9=Não sabe)
colunas_sem_filtro_789 = [
    #"id_domicilio",
    #PESO_AMOSTRAL,
    RAZAO_RENDA_POBREZA,
    CATEGORIA_RAZAO_POBREZA,
    PESO_LIBRAS,
    TEMPO_MORADIA_ANOS,
    QTD_ANOS_ESTADOS_UNIDOS,
]



for col in df_limpo.columns:
    if col not in colunas_sem_filtro_789:
        df_limpo[col] = df_limpo[col].replace(
            {7: np.nan, 8: np.nan, 9: np.nan, 7.0: np.nan, 8.0: np.nan, 9.0: np.nan}
        )

# Sentinelas específicas da variável contínua peso
if "peso_libras" in df_limpo.columns:
    df_limpo["peso_libras"] = df_limpo["peso_libras"].replace(
        {996: np.nan, 997: np.nan, 998: np.nan, 999: np.nan}
    )

# 2. Conversão das 38 variáveis binárias (CDC: 1=Sim, 2=Não -> 1=Sim, 0=Não)
vars_binarias = [
    COJUGE_TRABALHA,
    NATURAL_ESTADOS_UNIDOS,
    FLG_IDOSO_FAMILIA,
    RENDA_INVESTIMENTOS,
    LIMITACAO_TRABALHO_POR_SAUDE,
    ASMA_ATIVA_ATUALMENTE,
    HISTORICO_HIPERTENSAO,
    HISTORICO_COLESTEROL,
    REMEDIO_COLESTEROL,
    HISTORICO_ARTRITE,
    HISTORICO_DIABETES,
    DOR_CRONICA,
    CONCUSSAO_SINTOMAS_POS_TRAUMA,
    VACINA_GRIPE_ULTIMOS_12M,
    VACINA_TETANUS_10A,
    VISITAS_EXAME_VISTA_12M,
    ULTIMA_VISITA_FOI_CHECKUP,
    HISTORICO_DEPRESSAO,
    REMEDIO_DEPRESSAO,
    HISTORICO_TRANSTORNO_ANSIEDADE,
    MED_ANSIEDADE,
    ANDAR_NECESSITA_AJUDA_OUTRA_PESSOA,
    FUMA_CIGARRO_ATUALMENTE,
    USA_CIGARRO_ELETRONICO_ATUALMENTE,
    JA_FUMANTE,
    SEXO_CONJUGE,
    SEPARACAO_LEGAL_CONJUGAL,
    TEVE_CANCER
]

for col in vars_binarias:
    if col in df_limpo.columns:
        df_limpo[col] = df_limpo[col].replace({1: 1, 2: 0, 1.0: 1, 2.0: 0})

if "sexo" in df_limpo.columns:
    df_limpo["sexo"] = df_limpo["sexo"].replace({1: 0, 2: 1, 1.0: 0, 2.0: 1})

# 3. Escalas Ordinais e Clínicas (Início em 0 = Ausência de sintoma)
# PHQ-8 e GAD-7: CDC 1 a 4 -> 0 a 3
escalas_phq_gad = [
    FREQ_SOLITÁRIO,
    FREQ_SENTIU_DEPRIMIDO_2SEM,
    FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM,
    FREQ_PREOCUPACAO_DESNECESSARIA_2SEM,
    FREQ_DIFICULDADE_DORMIR_2SEM,
    FREQ_CANSASO_2SEM,
    FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM,
    FREQ_AUTOAVALIACAO_NEGTIVA_2SEM,
    FREQ_DIFICULDADE_CONCENTRACAO_2SEM,
    FREQ_ANSIEDADE_2SEM,
    FREQ_DIFICULDADE_DESCANSAR_2SEM,
    FREQ_AGITACAO_INQUIETA,
]
for col in escalas_phq_gad:
    if col in df_limpo.columns:
        df_limpo[col] = df_limpo[col].replace(
            {1: 0, 2: 1, 3: 2, 4: 3, 1.0: 0, 2.0: 1, 3.0: 2, 4.0: 3}
        )

# Frequência geral de sintomas (CDC: 1=Diariamente a 5=Nunca -> Inversão: 0=Nunca a 4=Diariamente)
for col in [FREQ_DEPRESSAO, FREQ_ANSIEDADE]:
    if col in df_limpo.columns:
        df_limpo[col] = df_limpo[col].replace(
            {5: 0, 4: 1, 3: 2, 2: 3, 1: 4, 5.0: 0, 4.0: 1, 3.0: 2, 2.0: 3, 1.0: 4}
        )

# Intensidade de sintomas (1=Pouco a 3=Muito -> 0=Pouco a 2=Muito)
for col in [INTENSIDADE_DEPRESSAO, INTENSIDADE_ANSIEDADE]:
    if col in df_limpo.columns:
        df_limpo[col] = df_limpo[col].replace(
            {1: 0, 2: 1, 3: 2, 1.0: 0, 2.0: 1, 3.0: 2}
        )

# Saúde geral (CDC: 1=Excelente a 5=Péssima -> Inversão:4=Excelente a 0=Péssima)
if AUTOAVALIACAO_SAUDE_GERAL in df_limpo.columns:
    df_limpo[AUTOAVALIACAO_SAUDE_GERAL] = df_limpo[
        AUTOAVALIACAO_SAUDE_GERAL
    ].replace({1: 4, 2: 3, 3: 2, 4: 1, 5: 0, 1.0: 4, 2.0: 3, 3.0: 2, 4.0: 1, 5.0: 0})

# Solidão (1=Sempre a 5=Nunca -> Inversão: 0=Nunca a 4=Sempre)
if FREQ_SOLITÁRIO in df_limpo.columns:
    df_limpo[FREQ_SOLITÁRIO] = df_limpo[FREQ_SOLITÁRIO].replace(
        {1: 4, 2: 3, 3: 2, 4: 1, 5: 0, 1.0: 4, 2.0: 3, 3.0: 2, 4.0: 1, 5.0: 0}
    )

# Suporte social (CDC: 1=Sempre a 5=Nunca -> Inversão: 0=Sempre a 4=Nunca/Sem suporte)
if SUPORTE_SOCIAL in df_limpo.columns:
    df_limpo[SUPORTE_SOCIAL] = df_limpo[SUPORTE_SOCIAL].replace(
        {1: 4, 2: 3, 3: 2, 4: 1, 5: 0, 1.0: 4, 2.0: 3, 3.0: 2, 4.0: 1, 5.0: 0}
    )

# Dificuldades motoras e cognitivas (1=Sem dificuldade a 4=Incapaz -> 0 a 3)
escalas_mobilidade = [
    ANDAR_DIFICULDADE_500M_SEM_APARELHO,
    ANDAR_DIFICULDADE_DEGRAUS_SEM_APARELHO,
    ANDAR_DIFICULDADE_500M_COM_APARELHO,
    ANDAR_DIFICULDADE_DEGRAUS_COM_APARELHO,
    FREQUENCIA_FALHA_MEMORIA,
    QTD_DIFICULDADE_COGNITIVA,
]
for col in escalas_mobilidade:
    if col in df_limpo.columns:
        df_limpo[col] = df_limpo[col].replace(
            {1: 0, 2: 1, 3: 2, 4: 3, 1.0: 0, 2.0: 1, 3.0: 2, 4.0: 3}
        )

# 4. One-Hot Encoding nas Categóricas Nominais Puras
vars_nominais = [
    REGIAO_GEOGRAFICA,
    CLASSIFICACAO_URBANO_RURAL,
    ESTADO_CIVIL,
    STATUS_PARENTAL,
    TIPO_POSSE_IMOVEL,
]
vars_nominais_presentes = [c for c in vars_nominais if c in df_limpo.columns]

df_processado = pd.get_dummies(
    df_limpo, columns=vars_nominais_presentes, drop_first=True, dtype=int
)

print(
    f"Processamento concluído: {df_processado.shape[0]:,} linhas e {df_processado.shape[1]} colunas geradas."
)

Processamento concluído: 24,215 linhas e 87 colunas geradas.


In [97]:

df_limpo[DIFICULDADE_LOCOMOCAO] = df_limpo[DIFICULDADE_LOCOMOCAO].replace(
{1: 0, 2: 1, 3: 2, 4: 3, 7:0, 8:0, 9:0, 1.0: 0, 2.0: 1, 3.0: 2, 4.0: 3, 7.0:0, 8.0:0, 9.0:0})

In [98]:
df_limpo[DIFICULDADE_LOCOMOCAO].value_counts()

DIFICULDADE_LOCOMOCAO
0.0    18521
1.0     4065
2.0     1315
3.0      310
Name: count, dtype: int64

In [99]:
porcentagem_nan(df_limpo)

[('QTD_CIGARRO_MES', 97.7575882717324), ('DIAS_FUMADOS_MES', 97.58827173239727), ('SEPARACAO_LEGAL_CONJUGAL', 97.53458600041297), ('QTD_CIGARROS_DIA', 93.78484410489366), ('ASMA_ATIVA_ATUALMENTE', 85.68655791864546), ('USA_CIGARRO_ELETRONICO_ATUALMENTE', 82.32913483378071), ('FREQUENCIA_FALHA_MEMORIA', 77.06793309931861), ('TIPO_DIFICULDADE_COGNITIVA', 73.17365269461078), ('QTD_VACINA_PNEUMONIA', 70.85277720421227), ('FUMA_CIGARRO_ATUALMENTE', 65.44290728887053), ('COJUGE_TRABALHA', 58.55461490811481), ('SEXO_CONJUGE', 58.249019202973365), ('ANO_VACINA_GRIPE', 54.239107990914725), ('ESCOLARIDADE_MAXIMA_FAMILIA', 52.41379310344828), ('INTENSIDADE_DEPRESSAO', 51.1418542225893), ('ESTADO_CIVIL', 30.613256246128433), ('INTENSIDADE_ANSIEDADE', 26.264711955399545), ('VACINA_TETANUS_10A', 6.343175717530457), ('RENDA_INVESTIMENTOS', 5.48007433409044), ('TIPO_POSSE_IMOVEL', 3.526739624199876), ('NATURAL_ESTADOS_UNIDOS', 3.2335329341317367), ('ESTADO_CIVIL_DECLARADO', 3.229403262440636), ('SUPOR

In [100]:
df_limpo[NATURAL_ESTADOS_UNIDOS]

0        1.0
1        1.0
2        1.0
3        1.0
4        1.0
        ... 
24210    1.0
24211    1.0
24212    1.0
24213    1.0
24214    1.0
Name: NATURAL_ESTADOS_UNIDOS, Length: 24215, dtype: float64

In [101]:
# ==============================================================================
# HIGIENIZAÇÃO SEM NAN: SEXO_CONJUGE, SEPARACAO_LEGAL_CONJUGAL E COJUGE_TRABALHA
# ==============================================================================

# 1. Recodificação de SEXO_CONJUGE
# 0 = Sem cônjuge | 1 = Masculino | 2 = Feminino | 9 = Recusou/Não sabe
if SEXO_CONJUGE in df_limpo.columns:
    df_limpo[SEXO_CONJUGE] = df_limpo[SEXO_CONJUGE].fillna(0)  # Pulo de fluxo
    df_limpo[SEXO_CONJUGE] = df_limpo[SEXO_CONJUGE].replace(
        {
            1: 1,
            2: 2,
            1.0: 1,
            2.0: 2,
            7: 9,
            8: 9,
            9: 9,
            7.0: 9,
            8.0: 9,
            9.0: 9,
        }
    )
    df_limpo[SEXO_CONJUGE] = df_limpo[SEXO_CONJUGE].astype(int)

# 2. Recodificação de SEPARACAO_LEGAL_CONJUGAL
# 0 = Não / Sem cônjuge | 1 = Sim | 9 = Recusou/Não sabe
if SEPARACAO_LEGAL_CONJUGAL in df_limpo.columns:
    df_limpo[SEPARACAO_LEGAL_CONJUGAL] = df_limpo[
        SEPARACAO_LEGAL_CONJUGAL
    ].fillna(0)
    df_limpo[SEPARACAO_LEGAL_CONJUGAL] = df_limpo[
        SEPARACAO_LEGAL_CONJUGAL
    ].replace(
        {
            1: 1,
            2: 0,
            1.0: 1,
            2.0: 0,
            7: 9,
            8: 9,
            9: 9,
            7.0: 9,
            8.0: 9,
            9.0: 9,
        }
    )
    df_limpo[SEPARACAO_LEGAL_CONJUGAL] = df_limpo[
        SEPARACAO_LEGAL_CONJUGAL
    ].astype(int)

# 3. Recodificação de COJUGE_TRABALHA
# 0 = Não / Sem cônjuge | 1 = Sim | 9 = Recusou/Não sabe
if COJUGE_TRABALHA in df_limpo.columns:
    df_limpo[COJUGE_TRABALHA] = df_limpo[COJUGE_TRABALHA].fillna(0)
    df_limpo[COJUGE_TRABALHA] = df_limpo[COJUGE_TRABALHA].replace(
        {
            1: 1,
            2: 0,
            1.0: 1,
            2.0: 0,
            7: 9,
            8: 9,
            9: 9,
            7.0: 9,
            8.0: 9,
            9.0: 9,
        }
    )
    df_limpo[COJUGE_TRABALHA] = df_limpo[COJUGE_TRABALHA].astype(int)

# Validação imediata: verifica se sobrou algum valor nulo nessas colunas
assert df_limpo[[SEXO_CONJUGE, SEPARACAO_LEGAL_CONJUGAL, COJUGE_TRABALHA]].isna().sum().sum() == 0, "Ainda existem NaNs nas variáveis de cônjuge!"

In [102]:
porcentagem_nan(df_limpo)

[('QTD_CIGARRO_MES', 97.7575882717324), ('DIAS_FUMADOS_MES', 97.58827173239727), ('QTD_CIGARROS_DIA', 93.78484410489366), ('ASMA_ATIVA_ATUALMENTE', 85.68655791864546), ('USA_CIGARRO_ELETRONICO_ATUALMENTE', 82.32913483378071), ('FREQUENCIA_FALHA_MEMORIA', 77.06793309931861), ('TIPO_DIFICULDADE_COGNITIVA', 73.17365269461078), ('QTD_VACINA_PNEUMONIA', 70.85277720421227), ('FUMA_CIGARRO_ATUALMENTE', 65.44290728887053), ('ANO_VACINA_GRIPE', 54.239107990914725), ('ESCOLARIDADE_MAXIMA_FAMILIA', 52.41379310344828), ('INTENSIDADE_DEPRESSAO', 51.1418542225893), ('ESTADO_CIVIL', 30.613256246128433), ('INTENSIDADE_ANSIEDADE', 26.264711955399545), ('VACINA_TETANUS_10A', 6.343175717530457), ('RENDA_INVESTIMENTOS', 5.48007433409044), ('TIPO_POSSE_IMOVEL', 3.526739624199876), ('NATURAL_ESTADOS_UNIDOS', 3.2335329341317367), ('ESTADO_CIVIL_DECLARADO', 3.229403262440636), ('SUPORTE_SOCIAL', 3.080735081561016), ('FREQ_SOLITÁRIO', 2.923807557299195), ('JA_FUMANTE', 2.7875283914928763), ('CONCUSSAO_SINTOMAS